# Live IG stream memory analysis

This notebook analyzes the bounded Windows IG-demo stream run using only the persisted `memory.csv` and `summary.json` artifacts. It separates startup warm-up from steady state and tests whether worker memory tracks SQLite file growth.

The notebook deliberately does **not** claim GC/native/cgroup attribution: this run recorded process RSS/private memory and SQLite file sizes, not PSS/smaps, GC generations, SQLite allocator counters, or Linux pressure signals.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('ggplot')

RUN_NAME = 'live-ig-stream-20260716T170448Z'
RUN_DIRECTORY_OVERRIDE = None  # Set to a Path/string to analyze another run.

candidates = []
if RUN_DIRECTORY_OVERRIDE:
    candidates.append(Path(RUN_DIRECTORY_OVERRIDE))
candidates.extend([
    Path('../artifacts') / RUN_NAME,
    Path('artifacts') / RUN_NAME,
    Path.cwd() / 'artifacts' / RUN_NAME,
])
RUN_DIRECTORY = next((p.resolve() for p in candidates if p.exists()), None)
if RUN_DIRECTORY is None:
    raise FileNotFoundError('Could not find the completed run directory. Set RUN_DIRECTORY_OVERRIDE.')

CSV_PATH = RUN_DIRECTORY / 'memory.csv'
SUMMARY_PATH = RUN_DIRECTORY / 'summary.json'
print(f'Run directory: {RUN_DIRECTORY}')
print(f'CSV: {CSV_PATH.stat().st_size:,} bytes')


In [ ]:
summary = json.loads(SUMMARY_PATH.read_text())
raw = pd.read_csv(CSV_PATH)
raw['timestamp'] = pd.to_datetime(raw['timestampUtc'], utc=True)
numeric_columns = ['pid', 'workingSetBytes', 'privateMemoryBytes', 'cpuSeconds', 'databaseBytes', 'walBytes', 'shmBytes']
raw[numeric_columns] = raw[numeric_columns].apply(pd.to_numeric, errors='coerce')
raw['rssMiB'] = raw['workingSetBytes'] / 2**20
raw['privateMiB'] = raw['privateMemoryBytes'] / 2**20
raw['databaseMiB'] = raw['databaseBytes'] / 2**20
raw['walMiB'] = raw['walBytes'] / 2**20
raw['shmMiB'] = raw['shmBytes'] / 2**20
worker = raw.loc[raw['process'].eq('Trading.Cli')].sort_values('timestamp').reset_index(drop=True)
if len(worker) < 2:
    raise ValueError('Expected at least two Trading.Cli samples.')
worker['elapsedHours'] = (worker['timestamp'] - worker['timestamp'].iloc[0]).dt.total_seconds() / 3600
worker['elapsedMinutes'] = worker['elapsedHours'] * 60
worker['sampleIntervalSeconds'] = worker['timestamp'].diff().dt.total_seconds()
worker['phase'] = np.where(worker['elapsedHours'] < 1, 'Warm-up (first hour)', 'Steady state (remaining five hours)')

print(f'Configured duration: {summary["DurationSeconds"] / 3600:.2f} h')
print(f'Rows: {len(raw):,}; Trading.Cli samples: {len(worker):,}')
print(f'Measured window: {worker["timestamp"].iloc[0]} to {worker["timestamp"].iloc[-1]}')
worker.head(3)

## 1. Integrity and headline metrics

The first hour is treated as warm-up because the process ramps from startup allocations into a stable plateau. The remaining observations are the primary steady-state test.

In [ ]:
def slope_per_hour(frame, column):
    x = frame['elapsedHours'].to_numpy(dtype=float)
    y = frame[column].to_numpy(dtype=float)
    return float(np.polyfit(x, y, 1)[0])

def pearson(frame, x, y):
    return float(frame[[x, y]].corr().iloc[0, 1])

warmup = worker.loc[worker['elapsedHours'] < 1]
steady = worker.loc[worker['elapsedHours'] >= 1]
headline = pd.DataFrame({
    'metric': [
        'Initial RSS (MiB)', 'Peak RSS (MiB)', 'Final RSS (MiB)',
        'Initial private (MiB)', 'Peak private (MiB)', 'Final private (MiB)',
        'Total SQLite growth (MiB)', 'Steady SQLite growth (MiB)',
        'Steady RSS delta (MiB)', 'Steady private delta (MiB)',
        'Steady RSS slope (MiB/hour)', 'Steady private slope (MiB/hour)',
        'Average sample interval (s)', 'Gaps over 15 seconds'
    ],
    'value': [
        worker['rssMiB'].iloc[0], worker['rssMiB'].max(), worker['rssMiB'].iloc[-1],
        worker['privateMiB'].iloc[0], worker['privateMiB'].max(), worker['privateMiB'].iloc[-1],
        worker['databaseMiB'].iloc[-1] - worker['databaseMiB'].iloc[0],
        steady['databaseMiB'].iloc[-1] - steady['databaseMiB'].iloc[0],
        steady['rssMiB'].iloc[-1] - steady['rssMiB'].iloc[0],
        steady['privateMiB'].iloc[-1] - steady['privateMiB'].iloc[0],
        slope_per_hour(steady, 'rssMiB'), slope_per_hour(steady, 'privateMiB'),
        worker['sampleIntervalSeconds'].dropna().mean(),
        int((worker['sampleIntervalSeconds'] > 15).sum()),
    ]
})
headline.style.format({'value': '{:.4f}'})

## 2. Worker memory evolution

This plot makes the key distinction visible: a startup ramp followed by a five-hour plateau. The rolling median reduces noise without hiding peaks.

In [ ]:
plot = worker.set_index('timestamp')[['rssMiB', 'privateMiB']].resample('1min').median()
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(plot.index, plot['rssMiB'], label='RSS', color='#1565c0', linewidth=1.8)
ax.plot(plot.index, plot['privateMiB'], label='Private memory', color='#ef6c00', linewidth=1.8)
warmup_boundary = worker['timestamp'].iloc[0] + pd.Timedelta(hours=1)
ax.axvline(warmup_boundary, color='#555', linestyle='--', linewidth=1.5, label='Warm-up boundary')
peak = worker.loc[worker['rssMiB'].idxmax()]
ax.scatter([peak['timestamp']], [peak['rssMiB']], color='#b71c1c', zorder=5)
ax.annotate(f'Peak {peak["rssMiB"]:.1f} MiB', (peak['timestamp'], peak['rssMiB']), xytext=(12, 12), textcoords='offset points', color='#b71c1c')
ax.set_title('Trading.Cli memory over the six-hour live stream')
ax.set_ylabel('MiB')
ax.legend(loc='best')
fig.autofmt_xdate()
plt.show()

## 3. SQLite growth versus worker memory

The secondary axis shows database/WAL/SHM file sizes. If database growth were driving unbounded resident memory, the two curves would continue rising together. Here the database grows in small steps while private memory is flat after warm-up.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(worker['timestamp'], worker['privateMiB'], color='#ef6c00', label='Private memory', linewidth=1.5)
ax1.plot(worker['timestamp'], worker['rssMiB'], color='#1565c0', alpha=0.55, label='RSS', linewidth=1.2)
ax1.set_ylabel('Worker memory (MiB)')
ax1.axvline(warmup_boundary, color='#555', linestyle='--', linewidth=1.2)
ax2 = ax1.twinx()
ax2.step(worker['timestamp'], worker['databaseMiB'], where='post', color='#2e7d32', label='SQLite file', linewidth=1.8)
ax2.step(worker['timestamp'], worker['walMiB'], where='post', color='#8e24aa', label='WAL', linewidth=1.2)
ax2.step(worker['timestamp'], worker['shmMiB'], where='post', color='#6d4c41', label='SHM', linewidth=1.2)
ax2.set_ylabel('SQLite files (MiB)')
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [line.get_label() for line in lines], loc='upper left')
ax1.set_title('SQLite file growth does not track post-warm-up private memory')
fig.autofmt_xdate()
plt.show()

## 4. Correlation and phase-aware regression

Correlation alone is not causality, especially when both variables move during startup. We therefore show both the full-run relationship and the steady-state relationship. A near-zero steady-state slope is the important result.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, frame, title in [(axes[0], worker, 'All samples'), (axes[1], steady, 'Steady state only')]:
    ax.scatter(frame['databaseMiB'], frame['privateMiB'], c=frame['elapsedHours'], cmap='viridis', s=8, alpha=0.5)
    coeff = np.polyfit(frame['databaseMiB'], frame['privateMiB'], 1)
    x = np.linspace(frame['databaseMiB'].min(), frame['databaseMiB'].max(), 50)
    ax.plot(x, coeff[0] * x + coeff[1], color='#b71c1c', linewidth=2, label=f'slope={coeff[0]:.2f} MiB/MiB')
    corr = pearson(frame, 'databaseMiB', 'privateMiB')
    ax.set_title(f'{title} (r={corr:.3f})')
    ax.set_xlabel('SQLite file size (MiB)')
    ax.legend()
axes[0].set_ylabel('Worker private memory (MiB)')
fig.suptitle('Memory versus SQLite size: phase-aware interpretation')
plt.tight_layout()
plt.show()

## 5. Hourly stability

Hourly medians and ranges expose whether memory keeps ratcheting upward. The first hour contains the warm-up; later medians should be stable if the process has reached a steady allocation regime.

In [ ]:
hourly = (worker.assign(hour=worker['elapsedHours'].floordiv(1).astype(int))
    .groupby('hour')
    .agg(rssMedian=('rssMiB', 'median'), rssMin=('rssMiB', 'min'), rssMax=('rssMiB', 'max'),
         privateMedian=('privateMiB', 'median'), privateMin=('privateMiB', 'min'), privateMax=('privateMiB', 'max'),
         databaseLast=('databaseMiB', 'last'), samples=('rssMiB', 'size'))
    .reset_index())
fig, ax = plt.subplots(figsize=(13, 5))
x = hourly['hour'].to_numpy()
ax.errorbar(x - 0.06, hourly['rssMedian'], yerr=[hourly['rssMedian'] - hourly['rssMin'], hourly['rssMax'] - hourly['rssMedian']], fmt='o-', capsize=4, label='RSS')
ax.errorbar(x + 0.06, hourly['privateMedian'], yerr=[hourly['privateMedian'] - hourly['privateMin'], hourly['privateMax'] - hourly['privateMedian']], fmt='o-', capsize=4, label='Private memory')
ax.set_xticks(x, [f'H{h}' for h in x])
ax.set_xlabel('Elapsed hour')
ax.set_ylabel('MiB (median with min/max range)')
ax.set_title('Hourly memory stability')
ax.legend()
plt.show()
hourly

## 6. What the sampler actually observed

The sampler follows the `dotnet run` process tree. This is useful for the local harness, but it is not host-wide process attribution and it does not provide cgroup/PSI evidence.

In [ ]:
family = (raw.groupby('process')
    .agg(rows=('pid', 'size'), uniquePids=('pid', 'nunique'), medianRssMiB=('rssMiB', 'median'), peakRssMiB=('rssMiB', 'max'),
         medianPrivateMiB=('privateMiB', 'median'), peakPrivateMiB=('privateMiB', 'max'))
    .sort_values('peakRssMiB', ascending=False))
display(family)

fig, ax = plt.subplots(figsize=(10, 5))
family[['medianRssMiB', 'peakRssMiB']].plot(kind='bar', ax=ax, color=['#90caf9', '#1565c0'])
ax.set_title('Observed process-family RSS')
ax.set_ylabel('MiB')
ax.set_xlabel('Observed process name')
ax.tick_params(axis='x', rotation=0)
plt.show()

## 7. Sampling quality and automated interpretation

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
intervals = worker['sampleIntervalSeconds'].dropna()
ax.hist(intervals, bins=25, color='#5e35b1', alpha=0.85)
ax.axvline(intervals.mean(), color='#b71c1c', linestyle='--', label=f'mean={intervals.mean():.2f}s')
ax.set_title('Trading.Cli sample interval distribution')
ax.set_xlabel('Seconds between worker samples')
ax.set_ylabel('Count')
ax.legend()
plt.show()

rss_steady_slope = slope_per_hour(steady, 'rssMiB')
private_steady_slope = slope_per_hour(steady, 'privateMiB')
db_steady_slope = slope_per_hour(steady, 'databaseMiB')
print('Evidence-based interpretation')
print(f'• Warm-up: RSS +{warmup["rssMiB"].iloc[-1] - warmup["rssMiB"].iloc[0]:.2f} MiB; private +{warmup["privateMiB"].iloc[-1] - warmup["privateMiB"].iloc[0]:.2f} MiB.')
print(f'• Steady state: RSS slope {rss_steady_slope:.3f} MiB/hour; private slope {private_steady_slope:.3f} MiB/hour.')
print(f'• SQLite: steady file slope {db_steady_slope:.3f} MiB/hour; WAL peak {worker["walMiB"].max():.3f} MiB.')
print(f'• RSS/private correlation with SQLite size: {pearson(worker, "databaseMiB", "rssMiB"):.3f} / {pearson(worker, "databaseMiB", "privateMiB"):.3f}.')
print('• Interpretation: no sustained SQLite-driven worker-memory growth is visible in this profile; GC/native/cgroup attribution remains outstanding.')